In [1]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "news_article_dataset.tsv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "articoder/news-dataset-for-news-bias-analysis",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

<ipython-input-1-c213d0f70412>:10: DeprecationWarning: load_dataset is deprecated and will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 23.2M/23.2M [00:00<00:00, 50.5MB/s]


First 5 records:    Unnamed: 0  Title of Headline Roundup description       Topics        Date  \
0        3513           Romney in London         NaN    Elections  2012-07-26   
1        3512     Romney's Overseas Trip         NaN    Elections  2012-08-01   
2        3511  Biden Comment Controversy         NaN    Elections  2012-08-16   
3        3510               Romney Taxes         NaN    Elections  2012-08-17   
4        3509  Skinnydipping Congressman         NaN  US Congress  2012-08-20   

                          url_story  \
0              /story/romney-london   
1      /story/romneys-overseas-trip   
2  /story/biden-comment-controversy   
3               /story/romney-taxes   
4  /story/skinnydipping-congressman   

                                    left_story_title  \
0                      Romney's Olympics false start   
1  Was Romney's trip 'a great success' or gaffe-f...   
2            Mission Impossible: Managing Joe Biden    
3  Obama Campaign Wants Romney To Rel

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8478 entries, 0 to 8477
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Unnamed: 0                 8478 non-null   int64 
 1   Title of Headline Roundup  8478 non-null   object
 2   description                4963 non-null   object
 3   Topics                     8478 non-null   object
 4   Date                       8478 non-null   object
 5   url_story                  8478 non-null   object
 6   left_story_title           8436 non-null   object
 7   center_story_title         7706 non-null   object
 8   right_story_title          8381 non-null   object
 9   right_story_url            8381 non-null   object
 10  right_story_text           8375 non-null   object
 11  center_story_url           7706 non-null   object
 12  center_story_text          7700 non-null   object
 13  left_story_url             8436 non-null   object
 14  left_sto

In [7]:
import pandas as pd

# Set pandas to display all rows
pd.set_option('display.max_rows', None)

In [8]:
print(df['Topics'].value_counts())

Topics
Politics                                744
Elections                               615
Economy and Jobs                        462
Coronavirus                             414
World                                   387
Middle East                             334
Donald Trump                            294
Immigration                             289
2024 Presidential Election              283
White House                             226
Supreme Court                           208
Violence in America                     203
Joe Biden                               190
Justice                                 152
Defense and Security                    138
Russia                                  133
Technology                              128
Media Industry                          126
Criminal Justice                        112
Healthcare                              110
General News                            106
Abortion                                104
Gun Control and Gun Right

In [ ]:
import pandas as pd

# Assuming your DataFrame is already loaded into `df`

# Rename relevant columns for consistency
df = df.rename(columns={
    'Title of Headline Roundup': 'roundup_title',
    'Topics': 'topics',
    'left_story_title': 'left_title',
    'left_story_text': 'left_text',
    'center_story_title': 'center_title',
    'center_story_text': 'center_text',
    'right_story_title': 'right_title',
    'right_story_text': 'right_text'
})

# Create Left Bias Subset
left_df = df[['roundup_title', 'topics', 'left_title', 'left_text']].dropna(subset=['left_title', 'left_text']).copy()
left_df['bias_label'] = 'left'
left_df.columns = ['roundup_title', 'topics', 'story_title', 'story_text', 'bias_label']

# Create Center Bias Subset
center_df = df[['roundup_title', 'topics', 'center_title', 'center_text']].dropna(subset=['center_title', 'center_text']).copy()
center_df['bias_label'] = 'center'
center_df.columns = ['roundup_title', 'topics', 'story_title', 'story_text', 'bias_label']

# Create Right Bias Subset
right_df = df[['roundup_title', 'topics', 'right_title', 'right_text']].dropna(subset=['right_title', 'right_text']).copy()
right_df['bias_label'] = 'right'
right_df.columns = ['roundup_title', 'topics', 'story_title', 'story_text', 'bias_label']

# Concatenate all subsets
final_df = pd.concat([left_df, center_df, right_df], ignore_index=True)

# Shuffle rows (optional)
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Preview
print(final_df.head())
print("\nClass balance:\n", final_df['bias_label'].value_counts())


                                       roundup_title             topics  \
0  Trump Considers Revoking Obama Intel Officials...  National Security   
1  How Political Polarization Paved the Way for t...       Polarization   
2  Venezuela Arrests 3 Americans Involved in Alle...       The Americas   
3  Abortion Bans to Take Effect in Florida, Missi...           Abortion   
4    US and China Agree to Suspend New Trade Tariffs     Foreign Policy   

                                         story_title  \
0  President Trump considers revoking security cl...   
1  Whether Robert Fico survives and resumes offic...   
2  US denies claim CIA plotted to kill Venezuela ...   
3  Court won't block Mississippi's abortion "trig...   
4  US-China trade war: Deal agreed to suspend new...   

                                          story_text bias_label  
0  President Donald Trump is exploring "mechanism...     center  
1  A few years after the dissolution of Czechoslo...       left  
2  The United 

In [ ]:
# Sort the combined dataframe by roundup title
final_df = final_df.sort_values(by='roundup_title').reset_index(drop=True)

final_df.head(50)

,roundup_title,topics,story_title,story_text,bias_label
0,"""Dream"" 50 Years Later",Civil Rights,Thousands Gather In D.C. To Mark 1963 Civil Ri...,People are assembling on the National Mall to ...,center
1,"""Dream"" 50 Years Later",Civil Rights,Remembering My Uncle's 'Dream',"Fifty years ago, a valiant group of people fro...",right
2,"""Dream"" 50 Years Later",Civil Rights,March In Washington To Continue Focus On Civil...,Alice Long planned months ago to use vacation ...,left
3,"""Good Shutdown"" in September?",Politics,"President Trump Calls for a ""Good Shutdown"" in...",President Donald Trump made a bold statement o...,right
4,"""Good Shutdown"" in September?",Politics,Trump's frustration with budget compromise has...,Congress looks set to enact a bipartisan spend...,left
5,"""Good Shutdown"" in September?",Politics,Trump: US ‘needs a good shutdown’,"President Trump on Tuesday called for a ""good ...",center
6,"""Skinny Repeal"" Motions Fails",US Senate,Obamacare repeal is dead for now. What could t...,The Senate’s effort to repeal and replace Obam...,center
7,"""Skinny Repeal"" Motions Fails",US Senate,Why Senate Republicans couldn’t repeal Obamacare,"In a stunning turn, Senate Republicans — in th...",left
8,"""Skinny Repeal"" Motions Fails",US Senate,Senate Fails To Pass Motion To Proceed On ‘Ski...,Senate Majority Leader Mitch McConnell failed ...,right
9,"""Trump University"" Documents Unsealed",Elections,'Trump University' Documents Put On Display Ag...,A federal judge released hundreds of pages of ...,center
